In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os
from skimage import morphology, segmentation
from scipy.ndimage import rotate
from skimage import segmentation, morphology
from matplotlib.colors import hsv_to_rgb
import Chain_Analysis_Functions as caf
from scipy.ndimage import binary_opening, binary_closing

## Step 1: Read in the Image and create the initial mask

In [ ]:
original_image_name = '57-connectors.jpg'
image_path = os.path.join(os.getcwd(), '0.3_20mT', original_image_name)
original_image = Image.open(image_path).convert('RGB')
mask = caf.create_binary_mask(image_path, brightness=1.0, contrast=1.0, saturation=1.0,
                           temperature=0, R_min=0, G_min=0, B_min=30, V_min=0.1,
                           method="adaptive", adaptive_block_size=10, adaptive_offset=0.01)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 8))

ax1.imshow(mask, cmap='gray')
ax1.set_title("Original Image")
ax1.axis("off")

# Top-left corner (works as-is)
ax2.imshow(mask[0:520, 0:520], cmap='gray')
ax2.set_title("Top left corner")
ax2.axis("off")

# Bottom-right corner — corrected slicing
ax3.imshow(mask[-520:, -520:], cmap='gray')
ax3.set_title("Bottom right corner")
ax3.axis("off")

plt.tight_layout()
plt.show()

## Step 2: Rotate the mask such that the particles align into perfect rows and columns

### Perform Rough Rotation

In [ ]:
angle = 0
rot_mask = caf.rotate(mask, angle, reshape=False, order=0, mode='constant', cval=0)
caf.display_mask(rot_mask, 20, 20)

#### Identify the appropriate rotation angle

In [ ]:
rot_angle_mask = morphology.remove_small_objects(rot_mask, min_size= 100) #Remove noise to keep it from interfering
rot_angle_mask, angle = caf.rotate_mask_until_balanced(rot_angle_mask, angle_step=0.1, tolerance=0.001, min_step=0.01, max_angle=5, search_rows = 50, search_columns = 3000)
print(angle)

In [ ]:
angle = 0.2686523437500001

#### Apply the rotation to the actual mask and confirm

In [ ]:
mask_rotated = caf.rotate(rot_mask, angle, reshape=False, order=0, mode='constant', cval=0)
caf.display_mask(mask_rotated)

### Crop the mask if necessary

In [ ]:
mask_new= mask_rotated[:-20,:]
caf.display_mask(mask_new,10,10)

#### Update the mask

In [ ]:
mask = mask_new
del mask_new

## Step 3: Identify the reference particle bounding box

In [ ]:
ref_mask = mask[2838:3137, 47:352]
caf.display_mask(ref_mask, 10, 10)

In [ ]:
ref_bbox = (2838,47,3137,352)
caf.show_reference(mask, ref_bbox)

## Step 4: Ensure reference crop is filled properly with set parameters

### Perform morphological Operations manually

In [ ]:
min_size = 20
opening_disk_size = 2
pad = 100
closing_disk_size1 = 35
larger_object_size = 200
max_row_gap1 = 50
max_row_gap = 10
max_column_gap = 50
opening_disk_size2 = 5
max_row_gap2 = 5
max_column_gap2 = 4
minr, minc, maxr, maxc = ref_bbox
ref_crop = mask[minr:maxr, minc:maxc].astype(bool)
cropmask = ref_crop.copy()
caf.display_mask(ref_crop,5,5)
ref_crop = morphology.remove_small_objects(ref_crop, min_size=min_size)
caf.display_mask(ref_crop,5,5)
ref_crop = binary_opening(ref_crop, structure = morphology.disk(opening_disk_size))
caf.display_mask(ref_crop,5,5)
ref_crop[210:220,35:43] = 0
ref_crop[258:263,15:20] = 0
ref_crop[0:35,:] = caf.fill_row_gaps2(ref_crop[0:35,:], max_gap = max_row_gap1)
ref_crop[-35:,:] = caf.fill_row_gaps2(ref_crop[-35:,:], max_gap = max_row_gap1)
ref_crop = caf.fill_row_gaps2(ref_crop, max_gap = max_row_gap)
caf.display_mask(ref_crop,5,5)
ref_crop = caf.fill_column_gaps(ref_crop, max_gap = max_column_gap)
caf.display_mask(ref_crop, 5, 5)
ref_crop = np.pad(ref_crop, pad_width = pad, mode='constant')
caf.display_mask(ref_crop,5,5)
ref_crop = binary_closing(ref_crop, structure = morphology.disk(closing_disk_size1))
caf.display_mask(ref_crop, 5, 5)
ref_crop = binary_closing(ref_crop, structure = morphology.disk(closing_disk_size1))
caf.display_mask(ref_crop, 5, 5)


ref_crop = ref_crop[pad:-pad, pad:-pad]
overlay_img = np.stack([ref_crop]*3, axis=-1).astype(float)

#Overlay in red (R=1, G=0, B=0)
overlay_img[cropmask.astype(bool), 0] = 1.0  # Red
overlay_img[cropmask.astype(bool), 1] = 0.0  # Green
overlay_img[cropmask.astype(bool), 2] = 0.0  # Blue

# Display result
plt.figure(figsize=(6, 6))
plt.imshow(overlay_img)
plt.title("Overlay Mask in Red")
plt.axis('off')
plt.show()


## Step 5: Ensure Last Data is Properly Identified

In [ ]:
caf.display_mask(mask)

In [ ]:
last_x, last_y = caf.check_last_row_and_column(mask, min_size = 100, last_row = 200, last_column = 200, plot = True)

## Step 6: Ensure array and particles are properly captured

In [ ]:
minr = 50
maxr = 349
minc = 45
maxc = 350
ref_bbox = (minr,minc,maxr,maxc)
caf.show_reference(mask, ref_bbox)

### Check X

In [ ]:
dx_offset = 168
stagger_x = False
stagger_x_frequency = 2

checking = True
num_rows = 1
num_cols = 15
particle_mask, debris_mask = caf.extract_particles_and_debris(mask, ref_bbox, ref_crop, min_size = 100, pad=10, max_gap = 10, dx_offset = dx_offset, dy_offset = 0,
                                                         stagger_x = stagger_x, stagger_y = False, stagger_y_frequency = 2, erode_pixels = 0, checking = checking,
                                                         check_last_row =  200, check_last_column = 200, num_rows = num_rows, num_cols = num_cols)

### Check Y

In [ ]:
dy_offset = 185
stagger_y = True
stagger_y_frequency = 4

checking = True
num_rows = 14
num_cols = 1
article_mask, debris_mask = caf.extract_particles_and_debris(mask, ref_bbox, ref_crop, min_size = 100, pad=10, max_gap = 10, dx_offset = dx_offset, dy_offset = dy_offset,
                                                         stagger_x = stagger_x, stagger_y = stagger_y, stagger_y_frequency = stagger_y_frequency, erode_pixels = 0, checking = checking,
                                                         check_last_row =  200, check_last_column = 200, num_rows = num_rows, num_cols = num_cols)

## Step 7: Apply all parameters and pull out particles from debris

In [ ]:
dx_offset = 168
stagger_x = False
stagger_x_frequency = 2

dy_offset = 185
stagger_y = True
stagger_y_frequency = 4

checking = False
erode_pixels = 2
num_rows = 14
num_cols = 15
particle_mask, debris_mask = caf.extract_particles_and_debris(mask, ref_bbox, ref_crop, min_size = 100, pad=10, max_gap = 10, dx_offset = dx_offset, dy_offset = dy_offset,
                                                          stagger_x = stagger_x, stagger_y = stagger_y, stagger_x_frequency = stagger_x_frequency, stagger_y_frequency = stagger_y_frequency,
                                                          erode_pixels = erode_pixels, checking = checking, check_last_row =  200, check_last_column = 200, num_rows = num_rows, num_cols = num_cols)

In [ ]:
caf.display_mask(particle_mask, 10, 10)

In [ ]:
caf.display_mask(particle_mask[50:360,50:360], 10, 10)

In [ ]:
caf.display_mask(debris_mask[:,:-400],10,10)

In [ ]:
caf.save_mask(particle_mask, image_path, '_0.png')
caf.save_mask(debris_mask, image_path,'_debris_0.png')

## Step 7: Identify bounds where particles are missing on the final wafer

### Step 7a: Load in cut wafer image and convert to a mask

In [ ]:
original_image_name = '0.3wt_strong_al_n1_VSM.jpg'
wafer_path = os.path.join(os.getcwd(), '0.3_20mT', original_image_name)
cropped_image = caf.show_original_image(wafer_path, x_size = 10, y_size = 10)

In [ ]:
caf.display_mask(mask,10,10)

In [ ]:
cut_mask = caf.cropped_image_to_mask(cropped_image, method="otsu")

### Step 7b: Rotate the Mask Roughly

In [ ]:
angle = 0
cut_mask_rotated = rotate(cut_mask, angle, reshape=False, order=0, mode='constant', cval=0)
caf.display_mask(cut_mask_rotated,10,10)

### Step 7c: Perform Initial Crop

In [ ]:
ymin = 100
ymax = 3300
xmin = 870
xmax = 4200
cut_mask = cut_mask_rotated[ymin:ymax, xmin:xmax]
caf.display_mask(cut_mask,10,10)

### Step 7d: Finely Rotate the Mask

In [ ]:
rot_angle_mask = ~cut_mask
rot_angle_cut_mask = morphology.remove_small_objects(rot_angle_mask, min_size= 200) #Remove noise to keep it from interfering
rot_angle_cut_mask = rotate(rot_angle_cut_mask, 180, reshape=False, order=0, mode='constant', cval=0)
rot_angle_cut_mask = rot_angle_cut_mask[:,500:]
caf.display_mask(rot_angle_cut_mask, 10,10)
rot_angle_cut_mask_2, cut_angle = caf.rotate_mask_until_balanced(rot_angle_cut_mask, angle_step=0.2, tolerance=0.005, min_step=0.01, max_angle=5, search_rows = 10, search_columns = 300)
print(cut_angle)
caf.display_mask(rot_angle_cut_mask_2, 10,10)

In [ ]:
angle = 0.4
cut_mask_rotated = rotate(cut_mask, angle, reshape=False, order=0, mode='constant', cval=0)
caf.display_mask(cut_mask_rotated, 10, 10)

### Step 7e: Identify the appropriate bounds in particle mask (roughly)

In [ ]:
original_image_name = '57-connectors.jpg'
image_path = os.path.join(os.getcwd(), '0.3_20mT', original_image_name)
original_image = Image.open(image_path).convert('RGB')
mask_name = '57-connectors_cleaned_mask.png'
particle_mask_name = os.path.join(os.getcwd(), '0.3_20mT', mask_name)
particle_mask = Image.open(particle_mask_name).convert('L')
particle_mask = np.array(particle_mask) == 255

In [ ]:
plus_x = 2450
plus_y = 2400
match_mask = particle_mask[:np.shape(cut_mask_rotated)[0]+plus_y, :np.shape(cut_mask_rotated)[1]+plus_x]
caf.display_mask(match_mask,10, 10)

In [ ]:
modify_rows_left = -15
modify_rows_right = 30
modify_cols_top = 25
modify_cols_bottom = -30
cut_mask2 =  caf.match_masks(~cut_mask_rotated, match_mask, modify_rows_left, modify_rows_right, modify_cols_top, modify_cols_bottom)

In [ ]:
caf.display_mask(cut_mask2, 10, 10)

### Step 7f: Add rows and columns to match with the particle mask

In [ ]:
cut_mask3 = caf.add_rows_to_match(~cut_mask2, particle_mask)

### Step 7g: Clean the mask to separate the area that has no particles

In [ ]:
cut_mask4 = caf.isolate_empty_space2(cut_mask3, remove_particle_size = 100000, small_object_size = 2000, max_column_gap1 = 100, max_row_gap = 200, max_filled_col_gap = 500,
                        max_filled_row_gap = 500, max_column_gap2 = 50, enhance_large_gap_size =200000, large_hole_threshold = 2000000, plot = True)

In [ ]:
caf.display_mask(cut_mask3.astype(np.uint8), 10, 10)
cut_mask5 = binary_closing(cut_mask3,structure = disk(20))
caf.display_mask(cut_mask5.astype(np.uint8), 10, 10)
cut_mask6 = caf.fill_small_holes2(cut_mask5, 50000, connectivity=1)
caf.display_mask(cut_mask6.astype(np.uint8), 10, 10)
cut_mask7 = ~caf.fill_column_gaps(~cut_mask6,30)
caf.display_mask(cut_mask7.astype(np.uint8), 10, 10)
cut_mask8= cut_mask7.copy()
cut_mask8[0:500,300:1200] = caf.fill_row_gaps2(cut_mask8[0:500,300:1200],100)
cut_mask8[0:500,300:1200] = caf.fill_column_gaps(cut_mask8[0:500,300:1200],50)
caf.display_mask(cut_mask8,10,10)
cut_mask9= cut_mask8.copy()
cut_mask9[500:,0:500] = caf.fill_column_gaps(cut_mask9[500:,0:500],300)
caf.display_mask(cut_mask9,10,10)
cut_mask10= cut_mask9.copy()
cut_mask10[3000:,4000:] = caf.fill_column_gaps(cut_mask10[3000:,4000:],150)
cut_mask10[3000:,4000:] = caf.fill_row_gaps2(cut_mask10[3000:,4000:],100)
caf.display_mask(cut_mask10,10,10)
cut_mask11 = caf.fill_small_holes2(cut_mask10,max_hole_size = 100000)
caf.display_mask(cut_mask11,10,10)
cut_mask12 = ~cut_mask11
cut_mask12[:,2500:] = caf.fill_row_gaps2(cut_mask12[:,2500:],500)
cut_mask12[2500:,:] = caf.fill_row_gaps2(cut_mask12[2500:,:],500)
cut_mask12 = caf.fill_column_gaps(cut_mask12,400)
caf.display_mask(cut_mask12,10,10)
outline = caf.mask_outline(cut_mask12)
caf.display_mask(outline,10,10)
cut_mask13 = cut_mask3 | cut_mask12
caf.display_mask(cut_mask13,10,10)
cut_mask14 = cut_mask13 &~outline
print('outline')
caf.display_mask(cut_mask14,10,10)
cut_mask15 = cut_mask14.copy()
cut_mask15[300:,:] = caf.row_threshold_mask(cut_mask15[300:,:],5550)
cut_mask15[:,:-500] = caf.column_threshold_mask(cut_mask15[:,:-500],5400)
caf.display_mask(cut_mask15,10,10)
cut_mask16 = ~cut_mask15.astype(bool)
cut_mask16[0:320,:] = caf.fill_column_gaps(cut_mask16[0:320,:],50)
cut_mask16 = ~cut_mask16
caf.display_mask(cut_mask16,10,10)
cut_mask17 = morphology.remove_small_objects(cut_mask16.astype(bool), min_size=5000000)
caf.display_mask(cut_mask17,10,10)
plot = True
small_object_size = 2000
max_column_gap1 = 100
cut_mask18 = ~cut_mask17
if plot:
    print('Invert Mask')
    caf.display_mask(cut_mask18.astype(np.uint8), 10, 10)

cut_mask18 = morphology.remove_small_objects(cut_mask18, min_size=small_object_size)
if plot:
    print('Remove tiny specs')
    caf.display_mask(cut_mask18.astype(np.uint8), 10, 10)

cut_mask18[300:-300,:] = caf.fill_column_gaps(cut_mask18[300:-300,:], max_gap = max_column_gap1)
if plot:
    print('Remove gaps in columns')
    caf.display_mask(cut_mask18.astype(np.uint8), 10, 10)

cut_mask18 = ~cut_mask18
cut_mask18 = caf.fill_row_gaps2(cut_mask18, max_gap = 40)
cut_mask18[:,0:300] = caf.fill_row_gaps2(cut_mask18[:,0:300], max_gap = 50)
cut_mask18 = ~cut_mask18
if plot:
    print('Remove thin beams')
    caf.display_mask(cut_mask18.astype(np.uint8), 10, 10)

max_row_gap = 200
max_filled_col_gap = 500
max_filled_row_gap = 500
max_column_gap2 = 50
enhance_large_gap_size =200000
large_hole_threshold = 2000000

cut_mask19 = cut_mask18.copy()
cut_mask19 = caf.fill_row_gaps2(cut_mask19, max_gap = max_row_gap)
if plot:
    print('Remove gaps in rows')
    caf.display_mask(cut_mask19.astype(np.uint8), 10, 10)

## Step 8: Filter particle mask using blank space from cut mask

In [ ]:
caf.display_mask(particle_mask,10,10)

In [ ]:
filtered_particle_mask = particle_mask & cut_mask19
caf.display_mask(filtered_particle_mask,10,10)

In [ ]:
filtered_debris_mask = debris_mask & cut_mask19
caf.display_mask(filtered_debris_mask,10,10)

In [ ]:
caf.save_mask(filtered_particle_mask,image_path,'_3.png')
caf.save_mask(filtered_debris_mask,image_path,'_debris_3.png')

# Identifying Chains in the Mask

## Below is a limited sample of the code that identifies all of the different chains within the mask. The full code can be run in "Run_chain_analysis.py"

In [ ]:
original_image_name = '57-connectors.jpg'
image_path = os.path.join(os.getcwd(), '0.3_20mT', original_image_name)
original_image = Image.open(image_path).convert('RGB')
mask_name = '57-connectors_cleaned_mask.png'
filtered_particle_mask_name = os.path.join(os.getcwd(), '0.3_20mT', mask_name)
filtered_particle_mask = Image.open(filtered_particle_mask_name).convert('L')
filtered_particle_mask = np.array(filtered_particle_mask) == 255
caf.display_mask(filtered_particle_mask)

In [ ]:
particle_bounds = (1000,1800,1000,1800)
region_counter, chain_mask, particle_region_masks = caf.label_mask_16(filtered_particle_mask, original_image, particle_bounds, disk_size=0, connectivity=1,
                                                                    branch_length_fraction=0.007, global_min_branch_length=2, min_region_size=200, debug_plots = False,
                                                                    prune=True, prune_branch_length=5, max_hole_size=20, vertical_prune_length = 4)

# Analyzing Chain Data

### Analyze the properties of a small subset of the identified chains and debris in the image

In [ ]:
mask_name = '57-connectors_cleaned_mask_debris.png'
debris_mask_name = os.path.join(os.getcwd(), '0.3_20mT', mask_name)
debris_mask = Image.open(debris_mask_name).convert('L')
debris_mask = np.array(debris_mask) == 255
debris_mask = debris_mask[0:500,0:500] #Look at a small number of particles at once

In [ ]:
df = caf.analyze_clusters(chain_mask, 1, angle_offset = None, fixed_angle = 0, plot = False, print_statement = False)

In [ ]:
print(np.unique(chain_mask))

In [ ]:
ex_mask = np.where(chain_mask == 50, 5, 0)
caf.display_mask(ex_mask, 10, 10)
ex_df = caf.analyze_clusters(ex_mask, 1, angle_offset = None, fixed_angle = 0, plot = True, print_statement = True)